<div style="display: flex; align-items: center; justify-content: center; flex-wrap: wrap;">
    <div style="flex: 1; max-width: 400px; display: flex; justify-content: center;">
        <img src="https://magic.novaims.unl.pt/media/1tdf2arr/ims25_horizontal__positivo_rgb.svg" style="max-width: 70%; height: auto; margin-top: 50px; margin-bottom: 50px;margin-left: 6rem;">
    </div>
    <div style="flex: 2; text-align: center; margin-top: 20px;margin-left: 6rem;">
        <div style="font-size: 28px; font-weight: bold; line-height: 1.2;">
            <span style="color: #FFCD41;">Thesis Project |</span> <span style="color: #F58228;">LLM-Powered Urban Exploration: A Framework for Adaptive Tourist and Mobility Route Planning</span>
        </div>
        <div style="font-size: 17px; font-weight: bold; margin-top: 10px;">
            2025 - 2026
        </div>
        <div style="font-size: 17px; font-weight: bold;">
            Master in Data Science and Advanced Analytics
        </div>
        <div style="margin-top: 20px;">
            <div>André Filipe Gomes Silvestre, 20240502</div>
        </div>
    </div>
</div>

<style>
@import url('https://fonts.cdnfonts.com/css/avenir-next-lt-pro?styles=29974');
</style>

<div style="background: linear-gradient(to right, #F58228, #FFCD41);
            padding: 15px; color: white; border-radius: 300px; text-align: center;">
    <center><h1 style="margin-left: 100px;margin-top: 10px; margin-bottom: 4px; color: white;
                       font-size: 32px; font-family: 'Avenir Next LT Pro', sans-serif;"><b>API Integration: IPMA, Metro, Carris, CP</b></h1></center>
</div>

<br><br>

## **📝 Notebook Overview**

This notebook demonstrates how to interact with various public APIs relevant to the thesis project. It includes examples of API calls, error handling, and data processing for:

1.  **[IPMA (Instituto Português do Mar e da Atmosfera)](https://api.ipma.pt/#about)**: Weather forecasts and warnings.
2.  **[Metro de Lisboa](https://app.metrolisboa.pt/status/getLinhas.php)**: Real-time service status.
3.  **[Carris Metropolitana](https://github.com/carrismetropolitana/api)**: Alerts, stops, lines, and routes.
4.  **[Comboios de Portugal (CP)](https://comboios.live/)**: Real-time train status and station information.

The goal is to establish a robust data collection pipeline for the intelligent tourist agent.

<br><br>

In [1]:
# Required libraries:
# pip install requests pandas

In [2]:
import requests
import pandas as pd
import json
import time
from datetime import datetime
from typing import Dict, List, Any, Optional

# Set pandas display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

## <span style="color: #ffffff;">1. IPMA API (Weather)</span>
<style>
@import url('https://fonts.cdnfonts.com/css/avenir-next-lt-pro?styles=29974');
</style>

<div style="background: transparent;
            padding: 10px; color: white; border-radius: 300px; text-align: center;
            border: 2px solid #F58228;">
    <center><h2 style="margin-left: 120px;margin-top: 10px; margin-bottom: 4px; color: #F58228;
                       font-size: 24px; font-family: 'Avenir Next LT Pro', sans-serif;"><b> 1. IPMA API (Weather)</b></h2></center>
</div>

<br><br>

The IPMA API provides weather forecasts and warnings. For this project, we focus on the Lisbon area.

**Endpoints:**
- **Warnings:** `https://api.ipma.pt/open-data/forecast/warnings/warnings_www.json`
- **Daily Forecast:** `https://api.ipma.pt/open-data/forecast/meteorology/cities/daily/{globalIdLocal}.json`
    - **Lisbon Global ID:** `1110600`

### **🔎 Dataset Attributes**

<center><b>Table 1 | </b> IPMA API Attributes. <br>

<br>

|       | **Field**                          | **Description**                                                                 | **Type**     |
|:-----:|:----------------------------------:|:--------------------------------------------------------------------------------|:------------:|
| **1** | `text`                             | Description of the warning (only for yellow/orange/red levels)                  | **`Text`**         |
| **2** | `awarenessTypeName`                | Type of warning (e.g., "Agitação Marítima", "Vento")                            | **`Text`**         |
| **3** | `awarenessLevelID`                 | Warning level color (green, yellow, orange, red)                                | **`Text`**         |
| **4** | `startTime`                        | Start time of the warning                                                       | **`DateTime`**     |
| **5** | `endTime`                          | End time of the warning                                                         | **`DateTime`**     |
| **6** | `precipitaProb`                    | Probability of precipitation (%)                                                | **`Numeric`**      |
| **7** | `tMin`                             | Minimum temperature (°C)                                                        | **`Numeric`**      |
| **8** | `tMax`                             | Maximum temperature (°C)                                                        | **`Numeric`**      |
| **9** | `predWindDir`                      | Predominant wind direction (e.g., N, SW)                                        | **`Text`**         |
| **10**| `forecastDate`                     | Date of the forecast                                                            | **`Date`**         |
</center>

<br><br>

In [3]:
def get_ipma_warnings() -> pd.DataFrame:
    """
    Fetches weather warnings from the IPMA API.

    Returns:
        pd.DataFrame: A DataFrame containing the warnings.
    """
    url = "https://api.ipma.pt/open-data/forecast/warnings/warnings_www.json"
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        data = response.json()
        
        # Filter for Lisbon area if needed, but the API returns a list. 
        # Assuming we want to see all warnings or filter later.
        # The prompt mentions "idAreaAviso": "LSB" for Lisbon in the context, but the example response shows "BGC".
        # We will return all and filter for demonstration.
        
        df = pd.DataFrame(data)
        return df
    except requests.exceptions.RequestException as e:
        print(f"\033[1mError fetching IPMA warnings:\033[0m {e}")
        return pd.DataFrame()

def get_ipma_forecast(global_id_local: int = 1110600) -> pd.DataFrame:
    """
    Fetches the daily weather forecast for a specific location.

    Args:
        global_id_local (int): The global ID of the location (default is 1110600 for Lisbon).

    Returns:
        pd.DataFrame: A DataFrame containing the daily forecast.
    """
    url = f"https://api.ipma.pt/open-data/forecast/meteorology/cities/daily/{global_id_local}.json"
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        data = response.json()
        
        if 'data' in data:
            df = pd.DataFrame(data['data'])
            # Add metadata columns
            df['globalIdLocal'] = data.get('globalIdLocal')
            df['dataUpdate'] = data.get('dataUpdate')
            return df
        else:
            print("\033[1mNo data found in IPMA forecast response.\033[0m")
            return pd.DataFrame()
            
    except requests.exceptions.RequestException as e:
        print(f"\033[1mError fetching IPMA forecast:\033[0m {e}")
        return pd.DataFrame()

# --- Example Usage ---
print("\n" + "="*80)
print("\033[1mIPMA Warnings\033[0m")
print("="*80)
df_warnings = get_ipma_warnings()
if not df_warnings.empty:
    # Display first 5 rows
    display(df_warnings.head())
else:
    print("No warnings data available.")

print("\n" + "="*80)
print("\033[1mIPMA Forecast (Lisbon)\033[0m")
print("="*80)
df_forecast = get_ipma_forecast()
if not df_forecast.empty:
    display(df_forecast.head())
else:
    print("\033[1mNo forecast data available.\033[0m")


IPMA Warnings


,text,awarenessTypeName,idAreaAviso,startTime,awarenessLevelID,endTime
0,"Queda de neve acima de 800/1000 m, descendo po...",Neve,BGC,2025-12-20T18:00:00,yellow,2025-12-21T09:00:00
1,,Agitação Marítima,BGC,2025-12-19T12:56:00,green,2025-12-22T12:00:00
2,,Nevoeiro,BGC,2025-12-19T12:56:00,green,2025-12-22T12:00:00
3,,Tempo Quente,BGC,2025-12-19T12:56:00,green,2025-12-22T12:00:00
4,,Tempo Frio,BGC,2025-12-19T12:56:00,green,2025-12-22T12:00:00



IPMA Forecast (Lisbon)


,precipitaProb,tMin,tMax,predWindDir,idWeatherType,classWindSpeed,longitude,forecastDate,classPrecInt,latitude,globalIdLocal,dataUpdate
0,100.0,10.5,14.9,N,6,2,-9.1286,2025-12-19,2.0,38.7660,1110600,2025-12-19T14:31:02
1,100.0,9.6,13.8,W,6,2,-9.1286,2025-12-20,2.0,38.7660,1110600,2025-12-19T14:31:02
2,100.0,7.5,11.6,NW,7,2,-9.1286,2025-12-21,1.0,38.7660,1110600,2025-12-19T14:31:02
3,82.0,7.5,15.1,W,7,2,-9.1286,2025-12-22,1.0,38.7660,1110600,2025-12-19T14:31:02
4,60.0,10.6,15.1,NW,3,2,-9.1286,2025-12-23,NaN,38.7660,1110600,2025-12-19T14:31:02


## <span style="color: #ffffff;">2. Metro de Lisboa API</span>
<style>
@import url('https://fonts.cdnfonts.com/css/avenir-next-lt-pro?styles=29974');
</style>

<div style="background: transparent;
            padding: 10px; color: white; border-radius: 300px; text-align: center;
            border: 2px solid #F58228;">
    <center><h2 style="margin-left: 120px;margin-top: 10px; margin-bottom: 4px; color: #F58228;
                       font-size: 24px; font-family: 'Avenir Next LT Pro', sans-serif;"><b> 2. Metro de Lisboa API</b></h2></center>
</div>

<br><br>

The Metro de Lisboa API provides the current status of the metro lines.

**Endpoint:** `https://app.metrolisboa.pt/status/getLinhas.php`

### **🔎 Dataset Attributes**

<center><b>Table 2 | </b> Metro de Lisboa API Attributes. <br>

<br>

|       | **Field**                          | **Description**                                                                 | **Type**     |
|:-----:|:----------------------------------:|:--------------------------------------------------------------------------------|:-----------:|
| **1** | `linha`                            | Name of the metro line (amarela, azul, verde, vermelha)                         | **`Text`**         |
| **2** | `status`                           | Operational status of the line (e.g., "Ok")                                     | **`Text`**         |
| **3** | `tipo_msg`                         | Message type code (internal use)                                                | **`Text`**         |

</center>

<br><br>

In [4]:
def get_metro_status() -> pd.DataFrame:
    """
    Fetches the status of Lisbon Metro lines.

    Returns:
        pd.DataFrame: A DataFrame containing the status of each line.
    """
    url = "https://app.metrolisboa.pt/status/getLinhas.php"
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        data = response.json()
        
        if 'resposta' in data:
            # The API returns a dictionary with line names as keys and status as values.
            # We need to transform this into a more usable format.
            status_data = data['resposta']
            
            # Extract lines and statuses
            lines = ['amarela', 'azul', 'verde', 'vermelha']
            formatted_data = []
            
            for line in lines:
                formatted_data.append({
                    'linha': line,
                    'status': status_data.get(line, 'Unknown'),
                    'tipo_msg': status_data.get(f'tipo_msg_{line[:2]}', '0') # e.g., tipo_msg_am
                })
            
            return pd.DataFrame(formatted_data)
        else:
            print("\033[1mUnexpected format in Metro API response.\033[0m")
            return pd.DataFrame()

    except requests.exceptions.RequestException as e:
        print(f"\033[1mError fetching Metro status:\033[0m {e}")
        return pd.DataFrame()

# --- Example Usage ---
print("\n" + "="*80)
print("\033[1mMetro de Lisboa Status\033[0m")
print("="*80)
df_metro = get_metro_status()
if not df_metro.empty:
    display(df_metro)
else:
    print("\033[1mNo metro data available.\033[0m")
    
# Static Information about Metro stations
# Date: 19/12/2025
# Site: https://dados.gov.pt/pt/datasets/r/be160193-3eaa-4799-bce1-672d7aa50c2f
def get_metro_stations() -> pd.DataFrame:
    """
    Fetches static information about Metro de Lisboa stations.

    Returns:
        pd.DataFrame: A DataFrame containing station information.
    """
    url = "https://services.arcgis.com/1dSrzEWVQn5kHHyK/arcgis/rest/services/POITransportes/FeatureServer/1/query?outFields=*&where=1%3D1&f=geojson"
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        data = response.json()
        
        df = pd.DataFrame(data)
        return df
    except requests.exceptions.RequestException as e:
        print(f"\033[1mError fetching Metro stations data:\033[0m {e}")
        return pd.DataFrame()
    
# --- Example Usage ---
print("\n" + "="*80)
print("\033[1mMetro de Lisboa Stations\033[0m")
print("="*80)
df_metro_stations = get_metro_stations()
if not df_metro_stations.empty:
    display(df_metro_stations.head())
else:
    print("\033[1mNo metro stations data available.\033[0m")


Metro de Lisboa Status


,linha,status,tipo_msg
0,amarela,Ok,0
1,azul,Ok,0
2,verde,Ok,0
3,vermelha,Ok,0



Metro de Lisboa Stations


,type,features
0,FeatureCollection,"{'type': 'Feature', 'id': 1, 'geometry': {'typ..."
1,FeatureCollection,"{'type': 'Feature', 'id': 2, 'geometry': {'typ..."
2,FeatureCollection,"{'type': 'Feature', 'id': 3, 'geometry': {'typ..."
3,FeatureCollection,"{'type': 'Feature', 'id': 4, 'geometry': {'typ..."
4,FeatureCollection,"{'type': 'Feature', 'id': 5, 'geometry': {'typ..."


## <span style="color: #ffffff;">3. Carris Metropolitana API</span>
<style>
@import url('https://fonts.cdnfonts.com/css/avenir-next-lt-pro?styles=29974');
</style>

<div style="background: transparent;
            padding: 10px; color: white; border-radius: 300px; text-align: center;
            border: 2px solid #F58228;">
    <center><h2 style="margin-left: 120px;margin-top: 10px; margin-bottom: 4px; color: #F58228;
                       font-size: 24px; font-family: 'Avenir Next LT Pro', sans-serif;"><b> 3. Carris Metropolitana API</b></h2></center>
</div>

<br><br>

Carris Metropolitana provides extensive data on bus services in the Lisbon Metropolitan Area. This API is open-source and provides network information in JSON or Protocol Buffers format, covering bus transit data for 15 of the 18 municipalities in the Lisbon metropolitan area.

**Base URL:** `https://api.carrismetropolitana.pt`

**Available Endpoints:**
- **Alerts:** `/v2/alerts` (Service alerts and disruptions)
- **Vehicles:** `/v2/vehicles` (Real-time vehicle positions)
- **Municipalities:** `/v2/municipalities` (Municipality information)
- **Stops:** `/v2/stops` (Stop information and real-time arrivals)
- **Lines:** `/v2/lines` (Line information)
- **Routes:** `/v2/routes` (Route information)
- **Patterns:** `/v2/patterns/:id` (Detailed journey patterns)
- **Shapes:** `/v2/shapes/:id` (Geographic shapes for routes)
- **ENCM:** `/v2/datasets/facilities/encm` (Customer service locations)
- **Schools:** `/v2/datasets/facilities/schools` (School locations)

### **🔎 Dataset Attributes**

<center><b>Table 3.1 | </b> Carris Metropolitana <strong>Alerts</strong> API Attributes. <br>

<br>

|       | **Field**                          | **Description**                                                                 | **Type**     |
|:-----:|:----------------------------------:|:--------------------------------------------------------------------------------|:-----------:|
| **1** | `alert_id`                         | Unique identifier for the alert                                                 | **`Text`**     |
| **2** | `cause`                            | Cause of the alert (e.g., "DEMONSTRATION", "ACCIDENT")                          | **`Text`**     |
| **3** | `effect`                           | Effect of the alert (e.g., "NO_SERVICE", "SIGNIFICANT_DELAYS")                  | **`Text`**     |
| **4** | `description_text`                 | Detailed description of the alert                                               | **`Text`**     |
| **5** | `header_text`                      | Short summary of the alert                                                      | **`Text`**     |
| **6** | `informed_entity`                  | List of affected routes, stops, or lines                                        | **`List`**     |
| **7** | `active_period`                    | Time periods when the alert is active                                           | **`List`**     |

</center>

<br>

<center><b>Table 3.2 | </b> Carris Metropolitana <strong>Vehicles</strong> API Attributes. <br>

<br>

|       | **Field**                          | **Description**                                                                 | **Type**     |
|:-----:|:----------------------------------:|:--------------------------------------------------------------------------------|:-----------:|
| **1** | `id`                               | Unique identifier for the vehicle                                               | **`Text`**     |
| **2** | `lat` / `lon`                      | Current geographical coordinates                                                | **`Numeric`**  |
| **3** | `speed`                            | Current speed of the vehicle                                                    | **`Numeric`**  |
| **4** | `heading`                          | Direction the vehicle is heading                                                | **`Numeric`**  |
| **5** | `trip_id`                          | Current trip identifier                                                         | **`Text`**     |
| **6** | `pattern_id`                       | Current pattern identifier                                                      | **`Text`**     |
| **7** | `timestamp`                        | Timestamp of last position update (milliseconds)                                | **`Numeric`**  |

</center>

<br>

<center><b>Table 3.3 | </b> Carris Metropolitana <strong>Stops</strong> API Attributes. <br>

<br>

|       | **Field**                          | **Description**                                                                 | **Type**     |
|:-----:|:----------------------------------:|:--------------------------------------------------------------------------------|:-----------:|
| **1** | `id`                               | Unique identifier for the stop                                                  | **`Text`**     |
| **2** | `name`                             | Full name of the stop                                                           | **`Text`**     |
| **3** | `short_name`                       | Short name of the stop                                                          | **`Text`**     |
| **4** | `lat` / `lon`                      | Geographical coordinates                                                        | **`Numeric`**  |
| **5** | `locality`                         | Locality where the stop is located                                              | **`Text`**     |
| **6** | `municipality_id` / `municipality_name` | Municipality information                                                  | **`Text`**     |
| **7** | `wheelchair_boarding`              | Wheelchair accessibility information                                            | **`Text`**     |
| **8** | `lines`                            | List of line IDs serving this stop                                              | **`List`**     |
| **9** | `routes`                           | List of route IDs serving this stop                                             | **`List`**     |
| **10**| `patterns`                         | List of pattern IDs serving this stop                                           | **`List`**     |
| **11**| `facilities`                       | Associated facilities (schools, ENCM, etc.)                                     | **`List`**     |

</center>

<br>

<center><b>Table 3.4 | </b> Carris Metropolitana <strong>Lines</strong> API Attributes. <br>

<br>

|       | **Field**                          | **Description**                                                                 | **Type**     |
|:-----:|:----------------------------------:|:--------------------------------------------------------------------------------|:-----------:|
| **1** | `id`                               | Unique identifier for the line                                                  | **`Text`**     |
| **2** | `short_name`                       | Short name/number of the line                                                   | **`Text`**     |
| **3** | `long_name`                        | Full descriptive name of the line                                               | **`Text`**     |
| **4** | `color`                            | Brand color of the line (hex code)                                              | **`Text`**     |
| **5** | `text_color`                       | Text color for the line (hex code)                                              | **`Text`**     |
| **6** | `municipalities`                   | List of municipalities served                                                   | **`List`**     |
| **7** | `localities`                       | List of localities served                                                       | **`List`**     |
| **8** | `routes`                           | List of route IDs for this line                                                 | **`List`**     |
| **9** | `patterns`                         | List of pattern IDs for this line                                               | **`List`**     |
| **10**| `facilities`                       | Associated facilities                                                           | **`List`**     |

</center>

<br><br>

In [5]:
def get_carris_data(endpoint: str, version: str = 'v2') -> pd.DataFrame:
    """
    Generic function to fetch data from Carris Metropolitana API.

    Args:
        endpoint (str): The API endpoint (e.g., 'alerts', 'stops', 'lines').
        version (str): API version (default: 'v2').

    Returns:
        pd.DataFrame: A DataFrame containing the requested data.
    
    Notes:
        - Implements retry logic with exponential backoff.
        - Uses timeouts to prevent hanging requests.
    """
    url = f"https://api.carrismetropolitana.pt/{version}/{endpoint}"
    max_retries = 3
    retry_delay = 2  # seconds
    
    for attempt in range(max_retries):
        try:
            response = requests.get(url, timeout=30)
            response.raise_for_status()
            data = response.json()
            
            # The API usually returns a list of objects directly
            df = pd.DataFrame(data)
            return df
        except requests.exceptions.RequestException as e:
            if attempt < max_retries - 1:
                print(f"\033[1mAttempt {attempt + 1} failed. Retrying in {retry_delay} seconds...\033[0m")
                time.sleep(retry_delay)
                retry_delay *= 2  # Exponential backoff
            else:
                print(f"\033[1mError fetching Carris {endpoint} after {max_retries} attempts:\033[0m {e}")
                return pd.DataFrame()

# --- Example Usage ---

# 1. Alerts
print("="*80)
print("\033[1m1. CARRIS ALERTS\033[0m")
print("="*80)
df_carris_alerts = get_carris_data('alerts')
if not df_carris_alerts.empty:
    print(f"\nTotal alerts: {len(df_carris_alerts)}")
    print(f"Columns: {list(df_carris_alerts.columns)}")
    display(df_carris_alerts.head())
else:
    print("No alerts available.")

# 2. Vehicles (Real-time)
print("\n" + "="*80)
print("\033[1m2. CARRIS VEHICLES (REAL-TIME)\033[0m")
print("="*80)
df_carris_vehicles = get_carris_data('vehicles')
if not df_carris_vehicles.empty:
    print(f"\nTotal vehicles: {len(df_carris_vehicles)}")
    print(f"Columns: {list(df_carris_vehicles.columns)}")
    display(df_carris_vehicles.head())
else:
    print("No vehicle data available.")

# 3. Municipalities
print("\n" + "="*80)
print("\033[1m3. CARRIS MUNICIPALITIES\033[0m")
print("="*80)
df_carris_municipalities = get_carris_data('municipalities')
if not df_carris_municipalities.empty:
    print(f"\nTotal municipalities: {len(df_carris_municipalities)}")
    print(f"Columns: {list(df_carris_municipalities.columns)}")
    display(df_carris_municipalities)
else:
    print("No municipality data available.")

# 4. Stops
print("\n" + "="*80)
print("\033[1m4. CARRIS STOPS (First 5)\033[0m")
print("="*80)
df_carris_stops = get_carris_data('stops')
if not df_carris_stops.empty:
    print(f"\nTotal stops: {len(df_carris_stops)}")
    print(f"Columns: {list(df_carris_stops.columns)}")
    display(df_carris_stops.head(5))
else:
    print("No stops data available.")

# 5. Lines
print("\n" + "="*80)
print("\033[1m5. CARRIS LINES (First 5)\033[0m")
print("="*80)
df_carris_lines = get_carris_data('lines')
if not df_carris_lines.empty:
    print(f"\nTotal lines: {len(df_carris_lines)}")
    print(f"Columns: {list(df_carris_lines.columns)}")
    display(df_carris_lines.head(5))
else:
    print("No lines data available.")

# 6. Routes
print("\n" + "="*80)
print("\033[1m6. CARRIS ROUTES (First 5)\033[0m")
print("="*80)
df_carris_routes = get_carris_data('routes')
if not df_carris_routes.empty:
    print(f"\nTotal routes: {len(df_carris_routes)}")
    print(f"Columns: {list(df_carris_routes.columns)}")
    display(df_carris_routes.head(5))
else:
    print("No routes data available.")

# 7. ENCM (Espaços navegante)
print("\n" + "="*80)
print("\033[1m7. CARRIS ENCM (CUSTOMER SERVICE LOCATIONS)\033[0m")
print("="*80)
df_carris_encm = get_carris_data('datasets/facilities/encm')
if not df_carris_encm.empty:
    print(f"\nTotal ENCM locations: {len(df_carris_encm)}")
    print(f"Columns: {list(df_carris_encm.columns)}")
    display(df_carris_encm)
else:
    print("No ENCM data available.")

# 8. Schools
print("\n" + "="*80)
print("\033[1m8. CARRIS SCHOOLS (First 5)\033[0m")
print("="*80)
df_carris_schools = get_carris_data('datasets/facilities/schools')
if not df_carris_schools.empty:
    print(f"\nTotal schools: {len(df_carris_schools)}")
    print(f"Columns: {list(df_carris_schools.columns)}")
    display(df_carris_schools.head(5))
else:
    print("No schools data available.")

1. CARRIS ALERTS

Total alerts: 120
Columns: ['active_period', 'cause', 'description_text', 'effect', 'header_text', 'informed_entity', 'url', 'alert_id', 'coordinates', 'image']


,active_period,cause,description_text,effect,header_text,informed_entity,url,alert_id,coordinates,image
0,"[{'end': 1766185663.902, 'start': 1766149663.9...",TRAFFIC_JAM,"{'translation': [{'language': 'pt', 'text': 'D...",SIGNIFICANT_DELAYS,"{'translation': [{'language': 'pt', 'text': '4...","[{'route_id': '[VURPT', 'line_id': '[VUR', 'tr...","{'translation': [{'language': 'pt-PT', 'text':...",EII5K,NaN,NaN
1,"[{'end': 1766185663.902, 'start': 1766149663.9...",TRAFFIC_JAM,"{'translation': [{'language': 'pt', 'text': 'D...",SIGNIFICANT_DELAYS,"{'translation': [{'language': 'pt', 'text': '4...","[{'route_id': '[VURPT', 'line_id': '[VUR', 'tr...","{'translation': [{'language': 'pt-PT', 'text':...",75JVN,NaN,NaN
2,"[{'end': 1766185663.902, 'start': 1766149663.9...",VEHICLE_ISSUE,"{'translation': [{'language': 'pt', 'text': 'D...",NO_SERVICE,"{'translation': [{'language': 'pt', 'text': '4...","[{'route_id': '[VURPT', 'line_id': '[VUR', 'tr...","{'translation': [{'language': 'pt-PT', 'text':...",49L3E,NaN,NaN
3,"[{'end': 1766185663.902, 'start': 1766149663.9...",VEHICLE_ISSUE,"{'translation': [{'language': 'pt', 'text': 'D...",NO_SERVICE,"{'translation': [{'language': 'pt', 'text': '4...","[{'route_id': '[VURPT', 'line_id': '[VUR', 'tr...","{'translation': [{'language': 'pt-PT', 'text':...",23E4J,NaN,NaN
4,"[{'end': 1766185663.902, 'start': 1766149663.9...",VEHICLE_ISSUE,"{'translation': [{'language': 'pt', 'text': 'D...",SIGNIFICANT_DELAYS,"{'translation': [{'language': 'pt', 'text': '4...","[{'route_id': '[VURPT', 'line_id': '[VUR', 'tr...","{'translation': [{'language': 'pt-PT', 'text':...",M27EQ,NaN,NaN



2. CARRIS VEHICLES (REAL-TIME)

Total vehicles: 1623
Columns: ['agency_id', 'bikes_allowed', 'capacity_seated', 'capacity_standing', 'capacity_total', 'contactless', 'id', 'wheelchair_accessible', 'license_plate', 'make', 'model', 'owner', 'propulsion', 'registration_date', 'bearing', 'block_id', 'current_status', 'event_id', 'lat', 'line_id', 'lon', 'pattern_id', 'route_id', 'schedule_relationship', 'shift_id', 'speed', 'stop_id', 'timestamp', 'trip_id', 'door_status', 'occupancy_estimated', 'occupancy_status', 'emission_class']


,agency_id,bikes_allowed,capacity_seated,capacity_standing,capacity_total,contactless,id,wheelchair_accessible,license_plate,make,model,owner,propulsion,registration_date,bearing,block_id,current_status,event_id,lat,line_id,lon,pattern_id,route_id,schedule_relationship,shift_id,speed,stop_id,timestamp,trip_id,door_status,occupancy_estimated,occupancy_status,emission_class
0,,False,NaN,NaN,NaN,False,|undefined,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,41,True,29.0,28.0,57.0,False,41|300,True,AC-45-FH,CaetanoBus,E. CITY GOLD,SCOTTURB,electricity,20141110,0.0,1_1325-11,STOPPED_AT,E468Y-41|300-1612_0_2_0800_0829_0_1,38.767357,1612,-9.298418,1612_0_2,1612_0,SCHEDULED,TP 1+1325,5.0,172486,1.766135e+09,[E468Y]1612_0_2_0800_0829_0_1,CLOSED,NaN,NaN,NaN
2,41,True,29.0,28.0,57.0,False,41|306,True,27-PE-51,Mercedes-Benz,OC 500LE 1830,VIMECA,diesel,20141110,163.0,1_1649-11,STOPPED_AT,E468Y-41|306-1601_1_1_0700_0729_0_1,38.680134,1601,-9.330452,1601_1_1,1601_1,SCHEDULED,1649,0.0,050007,1.766132e+09,[E468Y]1601_1_1_0700_0729_0_1,OPEN,NaN,NaN,NaN
3,41,True,29.0,28.0,57.0,False,41|308,True,84-PF-69,Mercedes-Benz,OC 500LE 1830,VIMECA,diesel,20141110,225.0,1_1642-11,STOPPED_AT,E468Y-41|308-1716_0_1_0630_0659_1_1,38.756954,1716,-9.248169,1716_0_1,1716_0,SCHEDULED,1642,0.0,030621,1.766129e+09,[E468Y]1716_0_1_0630_0659_1_1,NaN,NaN,NaN,NaN
4,41,True,29.0,28.0,57.0,False,41|311,True,84-PF-76,Mercedes-Benz,OC 500LE 1830,VIMECA,diesel,20111006,80.0,1_1244-11,IN_TRANSIT_TO,E468Y-41|311-1236_0_1_1430_1459_0_1,38.767384,1236,-9.298410,1236_0_1,1236_0,SCHEDULED,1260,0.0,172498,1.766158e+09,[E468Y]1236_0_1_1430_1459_0_1,NaN,NaN,NaN,NaN



3. CARRIS MUNICIPALITIES
Attempt 1 failed. Retrying in 2 seconds...
Attempt 2 failed. Retrying in 4 seconds...
Error fetching Carris municipalities after 3 attempts: 404 Client Error: Not Found for url: https://api.carrismetropolitana.pt/v2/municipalities
No municipality data available.

4. CARRIS STOPS (First 5)

Total stops: 12702
Columns: ['district_id', 'facilities', 'id', 'lat', 'line_ids', 'lon', 'long_name', 'municipality_id', 'pattern_ids', 'region_id', 'route_ids', 'short_name', 'tts_name', 'wheelchair_boarding']


...


5. CARRIS LINES (First 5)

Total lines: 715
Columns: ['color', 'district_ids', 'facilities', 'id', 'locality_ids', 'long_name', 'municipality_ids', 'pattern_ids', 'region_ids', 'route_ids', 'short_name', 'stop_ids', 'text_color', 'tts_name']


,color,district_ids,facilities,id,locality_ids,long_name,municipality_ids,pattern_ids,region_ids,route_ids,short_name,stop_ids,text_color,tts_name
0,#C61D23,[],[],1001,[],Alfragide (Estr Seminario) - Reboleira (Estação),[],"[1001_0_2, 1001_0_1]",[],[1001_0],1001,[],#FFFFFF,Linha 1001 com percurso Alfragide ( Estrada Se...
1,#C61D23,[],[],1002,[],Reboleira (Estação) | Circular via Alfragide (...,[],[1002_0_3],[],[1002_0],1002,[],#FFFFFF,Linha 1002 com percurso Reboleira ( - Estaçaão...
2,#C61D23,[],[],1003,[],Amadora (Estação Norte) - Amadora Este (Metro),[],"[1003_0_2, 1003_0_1]",[],[1003_0],1003,[],#FFFFFF,Linha 1003 com percurso Amadora ( - Estaçaão N...
3,#C61D23,[],[],1004,[],Amadora (Estação Norte) | Circular via Moinhos...,[],[1004_0_3],[],[1004_0],1004,[],#FFFFFF,Linha 1004 com percurso Amadora ( - Estaçaão N...
4,#C61D23,[],[],1005,[],Amadora (Estação Norte) - UBBO,[],"[1005_2_3, 1005_1_2, 1005_0_2, 1005_0_1]",[],"[1005_2, 1005_1, 1005_0]",1005,[],#FFFFFF,Linha 1005 com percurso Amadora ( - Estaçaão N...



6. CARRIS ROUTES (First 5)

Total routes: 944
Columns: ['color', 'district_ids', 'facilities', 'id', 'line_id', 'locality_ids', 'long_name', 'municipality_ids', 'pattern_ids', 'region_ids', 'short_name', 'stop_ids', 'text_color', 'tts_name']


,color,district_ids,facilities,id,line_id,locality_ids,long_name,municipality_ids,pattern_ids,region_ids,short_name,stop_ids,text_color,tts_name
0,#C61D23,[],[],1001_0,1001,[],Alfragide (Estr Seminario) - Reboleira (Estação),[],"[1001_0_2, 1001_0_1]",[],1001,[],#FFFFFF,Linha 1001 com percurso Alfragide ( Estrada Se...
1,#C61D23,[],[],1002_0,1002,[],Reboleira (Estação) | Circular via Alfragide (...,[],[1002_0_3],[],1002,[],#FFFFFF,Linha 1002 com percurso Reboleira ( - Estaçaão...
2,#C61D23,[],[],1003_0,1003,[],Amadora (Estação Norte) - Amadora Este (Metro),[],"[1003_0_2, 1003_0_1]",[],1003,[],#FFFFFF,Linha 1003 com percurso Amadora ( - Estaçaão N...
3,#C61D23,[],[],1004_0,1004,[],Amadora (Estação Norte) via Moinhos da Funchei...,[],[1004_0_3],[],1004,[],#FFFFFF,Linha 1004 com percurso Amadora ( - Estaçaão N...
4,#C61D23,[],[],1005_0,1005,[],Amadora (Estação Norte) - UBBO,[],"[1005_0_2, 1005_0_1]",[],1005,[],#FFFFFF,Linha 1005 com percurso Amadora ( - Estaçaão N...



7. CARRIS ENCM (CUSTOMER SERVICE LOCATIONS)
Attempt 1 failed. Retrying in 2 seconds...
Attempt 2 failed. Retrying in 4 seconds...
Error fetching Carris datasets/facilities/encm after 3 attempts: 404 Client Error: Not Found for url: https://api.carrismetropolitana.pt/v2/datasets/facilities/encm
No ENCM data available.

8. CARRIS SCHOOLS (First 5)
Attempt 1 failed. Retrying in 2 seconds...
Attempt 2 failed. Retrying in 4 seconds...
Error fetching Carris datasets/facilities/schools after 3 attempts: 404 Client Error: Not Found for url: https://api.carrismetropolitana.pt/v2/datasets/facilities/schools
No schools data available.


In [6]:
def get_carris_stop_realtime(stop_id: str) -> pd.DataFrame:
    """
    Fetches real-time arrival estimations for a specific stop.

    Args:
        stop_id (str): The unique identifier for the stop.

    Returns:
        pd.DataFrame: A DataFrame containing real-time arrival information.
    
    Notes:
        - Returns scheduled, estimated, and observed arrival times.
        - Includes vehicle ID and trip information.
    """
    url = f"https://api.carrismetropolitana.pt/v2/stops/{stop_id}/realtime"
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        data = response.json()
        
        df = pd.DataFrame(data)
        return df
    except requests.exceptions.RequestException as e:
        print(f"\033[1mError fetching real-time data for stop {stop_id}:\033[0m {e}")
        return pd.DataFrame()

def get_carris_pattern(pattern_id: str) -> Dict:
    """
    Fetches detailed information for a specific pattern.

    Args:
        pattern_id (str): The unique identifier for the pattern.

    Returns:
        Dict: A dictionary containing pattern information including path and trips.
    
    Notes:
        - Pattern objects are large and contain full schedule data.
        - Returns nested structure with stops, trips, and timing information.
    """
    url = f"https://api.carrismetropolitana.pt/v2/patterns/{pattern_id}"
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        data = response.json()
        return data
    except requests.exceptions.RequestException as e:
        print(f"\033[1mError fetching pattern {pattern_id}:\033[0m {e}")
        return {}

def get_carris_shape(shape_id: str) -> Dict:
    """
    Fetches geographic shape data for a specific route shape.

    Args:
        shape_id (str): The unique identifier for the shape.

    Returns:
        Dict: A dictionary containing shape points in GTFS and GeoJSON format.
    
    Notes:
        - Includes extension (total distance) in meters.
        - GeoJSON format is suitable for mapping applications.
    """
    url = f"https://api.carrismetropolitana.pt/v2/shapes/{shape_id}"
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        data = response.json()
        return data
    except requests.exceptions.RequestException as e:
        print(f"\033[1mError fetching shape {shape_id}:\033[0m {e}")
        return {}

# --- Example Usage: Real-time Stop Data ---
print("\n" + "="*80)
print("\033[1m9. EXAMPLE: REAL-TIME ARRIVALS FOR A SPECIFIC STOP\033[0m")
print("="*80)

# Get a sample stop ID from the stops data
if not df_carris_stops.empty:
    sample_stop_id = df_carris_stops.iloc[0]['id']
    print(f"\n\033[1mFetching real-time data for stop:\033[0m {sample_stop_id}")
    
    # Safely access 'name' column if it exists
    if 'name' in df_carris_stops.columns:
        print(f"\033[1mStop name:\033[0m {df_carris_stops.iloc[0]['name']}")
    
    df_realtime = get_carris_stop_realtime(sample_stop_id)
    if not df_realtime.empty:
        print(f"\nColumns: {list(df_realtime.columns)}")
        display(df_realtime)
    else:
        print("\033[1mNo real-time arrivals available for this stop.\033[0m")
else:
    print("\033[1mCannot fetch real-time data: No stops data available.\033[0m")

# --- Example Usage: Pattern Data ---
print("\n" + "="*80)
print("\033[1m10. EXAMPLE: PATTERN DETAILS\033[0m")
print("="*80)

# Initialize pattern_data for later use
pattern_data = {}

# Get a sample pattern ID from the lines data
if not df_carris_lines.empty and 'patterns' in df_carris_lines.columns:
    sample_patterns = df_carris_lines.iloc[0]['patterns']
    if sample_patterns and len(sample_patterns) > 0:
        sample_pattern_id = sample_patterns[0]
        print(f"\n\033[1mFetching pattern data for:\033[0m {sample_pattern_id}")
        print(f"\033[1mLine:\033[0m {df_carris_lines.iloc[0]['short_name']} - {df_carris_lines.iloc[0]['long_name']}")
        
        pattern_data = get_carris_pattern(sample_pattern_id)
        if pattern_data:
            print(f"\n\033[1mPattern ID:\033[0m {pattern_data.get('id')}")
            print(f"\033[1mHeadsign:\033[0m {pattern_data.get('headsign')}")
            print(f"\033[1mValid on dates (first 5):\033[0m {pattern_data.get('valid_on', [])[:5]}")
            print(f"\033[1mNumber of stops in path:\033[0m {len(pattern_data.get('path', []))}")
            print(f"\033[1mNumber of trips:\033[0m {len(pattern_data.get('trips', []))}")
            
            # Show first 3 stops in the path
            if 'path' in pattern_data and pattern_data['path']:
                print("\n\033[1mFirst 3 stops in path:\033[0m")
                df_path = pd.DataFrame(pattern_data['path'][:3])
                display(df_path)
        else:
            print("No pattern data available.")
    else:
        print("No patterns available for this line.")
else:
    print("Cannot fetch pattern data: No lines data available.")
    
# --- Example Usage: Shape Data ---
print("\n" + "="*80)
print("\033[1m11. EXAMPLE: SHAPE GEOMETRY\033[0m")
print("="*80)

# Try to get shape from pattern data
if pattern_data and 'shape_id' in pattern_data:
    sample_shape_id = pattern_data['shape_id']
    print(f"\n\033[1mFetching shape data for:\033[0m {sample_shape_id}")
    
    shape_data = get_carris_shape(sample_shape_id)
    if shape_data:
        print(f"\n\033[1mShape ID:\033[0m {shape_data.get('id')}")
        print(f"\033[1mExtension:\033[0m {shape_data.get('extension')} meters")
        print(f"\033[1mNumber of points:\033[0m {len(shape_data.get('points', []))}")
        
        # Show first 3 points
        if 'points' in shape_data and shape_data['points']:
            print("\n\033[1mFirst 3 coordinate points:\033[0m")
            df_points = pd.DataFrame(shape_data['points'][:3])
            display(df_points)
            
        # Show GeoJSON structure (first 3 coordinates)
        if 'geojson' in shape_data:
            geojson = shape_data['geojson']
            print("\n\033[1mGeoJSON type:\033[0m", geojson.get('type'))
            print("\033[1mGeometry type:\033[0m", geojson.get('geometry', {}).get('type'))
            print("\033[1mFirst 3 coordinates:\033[0m", geojson.get('geometry', {}).get('coordinates', [])[:3])
    else:
        print("No shape data available.")
else:
    print("Cannot fetch shape data: No pattern data available.")


9. EXAMPLE: REAL-TIME ARRIVALS FOR A SPECIFIC STOP

Fetching real-time data for stop: 010001
Error fetching real-time data for stop 010001: 404 Client Error: Not Found for url: https://api.carrismetropolitana.pt/v2/stops/010001/realtime
No real-time arrivals available for this stop.

10. EXAMPLE: PATTERN DETAILS
Cannot fetch pattern data: No lines data available.

11. EXAMPLE: SHAPE GEOMETRY
Cannot fetch shape data: No pattern data available.


### **Additional Carris Endpoints**

The following endpoints provide more detailed information and are typically used for specific queries:

- **Stop Real-time Arrivals:** `GET /stops/:id/realtime` (Real-time arrival estimations for a specific stop)
- **Single Pattern:** `GET /patterns/:id` (Detailed information about a specific pattern, including path and trips)
- **Single Shape:** `GET /shapes/:id` (Geographic shape data in GTFS and GeoJSON format)
- **Single Municipality:** `GET /municipalities/:id` (Specific municipality information)
- **Single Stop:** `GET /stops/:id` (Detailed information about a specific stop)
- **Single Line:** `GET /lines/:id` (Detailed information about a specific line)
- **Single Route:** `GET /routes/:id` (Detailed information about a specific route)
- **Single ENCM:** `GET /datasets/facilities/encm/:id` (Specific customer service location)
- **Single School:** `GET /datasets/facilities/schools/:id` (Specific school information)

## <span style="color: #ffffff;">4. Comboios de Portugal (CP) API</span>
<style>
@import url('https://fonts.cdnfonts.com/css/avenir-next-lt-pro?styles=29974');
</style>

<div style="background: transparent;
            padding: 10px; color: white; border-radius: 300px; text-align: center;
            border: 2px solid #F58228;">
    <center><h2 style="margin-left: 120px;margin-top: 10px; margin-bottom: 4px; color: #F58228;
                       font-size: 24px; font-family: 'Avenir Next LT Pro', sans-serif;"><b> 4. Comboios de Portugal (CP) API</b></h2></center>
</div>

<br><br>

The CP API (via comboios.live) provides real-time information about trains and stations.

**Endpoints:**
- **Stations:** `https://comboios.live/api/stations`
- **Vehicles (Real-time):** `https://comboios.live/api/vehicles`

### **🔎 Dataset Attributes**

<center><b>Table 4 | </b> CP API Attributes. <br>

<br>

|       | **Field**                          | **Description**                                                                 | **Type**     |
|:-----:|:----------------------------------:|:--------------------------------------------------------------------------------|:-----------:|
| **1** | `trainNumber`                      | Unique identifier for the train                                                 | **`Numeric`**      |
| **2** | `runDate`                          | Date of the train run                                                           | **`Date`**         |
| **3** | `delay`                            | Current delay in seconds                                                        | **`Numeric`**      |
| **4** | `status`                           | Status of the train (e.g., "IN_TRANSIT")                                        | **`Text`**         |
| **5** | `latitude` / `longitude`           | Current geographical position of the train                                      | **`Numeric`**    |
| **6** | `designation` (Station)            | Name of the station                                                             | **`Text`**         |
| **7** | `code` (Station)                   | Station code                                                                    | **`Text`**         |
</center>

</center>

<br><br>

In [7]:
def get_cp_stations() -> pd.DataFrame:
    """
    Fetches the list of CP stations.

    Returns:
        pd.DataFrame: A DataFrame containing station information.
    """
    url = "https://comboios.live/api/stations"
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        data = response.json()
        
        if 'stations' in data:
            df = pd.DataFrame(data['stations'])
            return df
        else:
            print("Unexpected format in CP Stations API response.")
            return pd.DataFrame()

    except requests.exceptions.RequestException as e:
        print(f"\033[1mError fetching CP stations:\033[0m {e}")
        return pd.DataFrame()

def get_cp_vehicles() -> pd.DataFrame:
    """
    Fetches real-time information about CP vehicles (trains).

    Returns:
        pd.DataFrame: A DataFrame containing vehicle information.
    """
    url = "https://comboios.live/api/vehicles"
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        data = response.json()
        
        if 'vehicles' in data:
            df = pd.DataFrame(data['vehicles'])
            return df
        else:
            print("Unexpected format in CP Vehicles API response.")
            return pd.DataFrame()

    except requests.exceptions.RequestException as e:
        print(f"\033[1mError fetching CP vehicles:\033[0m {e}")
        return pd.DataFrame()

# --- Example Usage ---
print("\n" + "="*80)
print("\033[1mCP Stations (First 5)\033[0m")
print("="*80)
df_cp_stations = get_cp_stations()
if not df_cp_stations.empty:
    display(df_cp_stations.head())
else:
    print("No station data available.")

print("\n" + "="*80)
print("\033[1mCP Vehicles (Real-time)\033[0m")
print("="*80)
df_cp_vehicles = get_cp_vehicles()
if not df_cp_vehicles.empty:
    display(df_cp_vehicles.head())
else:
    print("No vehicle data available.")


CP Stations (First 5)


,code,designation,latitude,longitude,region,railways
0,94-52001,Abrantes,39.4401931762695,-8.19432544708252,None,[]
1,94-36046,Ademia,40.2509269714355,-8.45034217834473,None,[]
2,94-18119,Afife,41.7753791809082,-8.86147117614746,None,[1]
3,94-61002,Agualva - Cacem,38.7664260864258,-9.29842472076416,None,[]
4,94-3046,Aguas Santas - Palmilheira,41.2017288208008,-8.55673980712891,None,[1]



CP Vehicles (Real-time)


,trainNumber,runDate,delay,lastStation,latitude,longitude,status,hasDisruptions,service,origin,destination
0,132,2025-12-19,433.0,94-36004,39.66580380030172,-8.498613598277283,IN_TRANSIT,False,"{'code': '1', 'designation': 'Alfa Pendular'}","{'code': '94-29157', 'designation': 'Braga'}","{'code': '94-30007', 'designation': 'Lisboa Sa..."
1,133,2025-12-19,-162.0,94-31039,40.16711480104076,-8.630324201742894,IN_TRANSIT,False,"{'code': '1', 'designation': 'Alfa Pendular'}","{'code': '94-30007', 'designation': 'Lisboa Sa...","{'code': '94-29157', 'designation': 'Braga'}"
2,135,2025-12-19,0.0,94-30007,38.71527399540608,-9.121403620116261,AT_ORIGIN,False,"{'code': '1', 'designation': 'Alfa Pendular'}","{'code': '94-30007', 'designation': 'Lisboa Sa...","{'code': '94-29157', 'designation': 'Braga'}"
3,184,2025-12-19,433.0,94-38000,40.516219171616726,-8.506299320230209,IN_TRANSIT,False,"{'code': '1', 'designation': 'Alfa Pendular'}","{'code': '94-31039', 'designation': 'Lisboa Or...","{'code': '94-73007', 'designation': 'Faro'}"
4,186,2025-12-19,0.0,94-73007,37.019742230824995,-7.940954601577709,AT_ORIGIN,False,"{'code': '1', 'designation': 'Alfa Pendular'}","{'code': '94-73007', 'designation': 'Faro'}","{'code': '94-2006', 'designation': 'Porto Camp..."


<div style="background: linear-gradient(to right, #F58228, #FFCD41);
            padding: 3px; color: white; border-radius: 900px; text-align: center;">
</div>

<br>

## **🏁 Conclusion**

1.  **IPMA**: Weather data is crucial for suggesting outdoor vs. indoor activities.
2.  **Metro de Lisboa**: Service status helps in avoiding delays.
3.  **Carris Metropolitana**: Comprehensive bus network data allows for detailed route planning.
4.  **CP**: Real-time train data supports regional mobility.

The functions defined here can be modularized and integrated into the main agent's codebase to provide real-time context for itinerary planning.